In [ ]:
# In this part, we get nodes from 7 RF textbooks and filter them. And node7.jsonl is the final file.

In [21]:
import re, json
from pathlib import Path

CUR_DIR = Path.cwd()          
OUTPUT_FILE = CUR_DIR / "node.jsonl"

re_img_md  = re.compile(r'!\[.*?\]\(.*?\)')           
re_figure  = re.compile(r'^\s*(Figure|Fig\.)\s*\d+', re.I)
re_subheading = re.compile(r'^(#+\s*\d+\.\d+(?:\.\d+)*\b.*)$', re.MULTILINE)

def process_md_file(path: Path):
    with path.open(encoding="utf-8") as f:
        content = f.read()

    lines = content.split('\n')
    output_lines = []
    in_section = False
    current_section = []
    
    for line in lines:
        line = line.strip()
        if not line:
            continue

        if re_subheading.match(line):
            if current_section:
                section_content = '\n'.join(current_section)
                cleaned_content = clean_content_simple(section_content)
                if cleaned_content:
                    output_lines.append(cleaned_content)
                current_section = []

            in_section = True
            current_section.append(f"<little_title>{line}")
        else:

            if in_section:
                current_section.append(line)

    if current_section:
        section_content = '\n'.join(current_section)
        cleaned_content = clean_content_simple(section_content)
        if cleaned_content:
            output_lines.append(cleaned_content)

    if not output_lines:
        cleaned_content = clean_content_simple(content)
        if cleaned_content:
            yield cleaned_content
    else:
        for content_text in output_lines:
            yield content_text

def clean_content_simple(text: str) -> str:
    if not text:
        return ""
    
    lines = []
    for line in text.split('\n'):
        line = re_img_md.sub('', line).strip()
        if not line:
            continue
        if re_figure.match(line):
            continue
        lines.append(line)
    
    return '\n'.join(lines)

with OUTPUT_FILE.open("w", encoding="utf-8") as fout:
    for md_path in CUR_DIR.glob("*.md"):
        print(f"deal with file: {md_path.name}")
        for para in process_md_file(md_path):
            fout.write(json.dumps({"text": para}, ensure_ascii=False) + "\n")

print(f"output dataset is: {OUTPUT_FILE}")

deal with file: razavi+chapter4-11.md
deal with file: markdown+(10).md
deal with file: markdown+(9).md
deal with file: wireless.md
deal with file: markdown+(8).md
deal with file: advanced.md
deal with file: razavi+chapter12.md
deal with file: razavi+chapter1-3.md
deal with file: Kazimierczu+chapter1-8.md
deal with file: kazimierczu+chapter9-12.md
output dataset is: /home/xy67/RFIC Chatbot/QAs from textbooks/node.jsonl


In [30]:
import re, json
from pathlib import Path

CUR_DIR = Path.cwd()
SPECIAL_FILE = "markdown+(11).md"
NODE_FILE = "node.jsonl"
MERGED_FILE = "node1.jsonl"

re_img_md  = re.compile(r'!\[.*?\]\(.*?\)')           
re_figure  = re.compile(r'^\s*(Figure|Fig\.)\s*\d+', re.I)
re_heading = re.compile(r'^(#+ .*)$', re.MULTILINE)

def process_special_file(path: Path):
    with path.open(encoding="utf-8") as f:
        content = f.read()

    lines = content.split('\n')
    output_lines = []
    current_section = []
    
    for line in lines:
        line = line.strip()
        if not line:
            continue

        if re_heading.match(line):
            if current_section:
                section_content = '\n'.join(current_section)
                cleaned_content = clean_content_simple(section_content)
                if cleaned_content:
                    output_lines.append(cleaned_content)
                current_section = []
            current_section.append(line)
        else:
            if current_section:
                current_section.append(line)

    if current_section:
        section_content = '\n'.join(current_section)
        cleaned_content = clean_content_simple(section_content)
        if cleaned_content:
            output_lines.append(cleaned_content)

    return output_lines

def clean_content_simple(text: str) -> str:
    if not text:
        return ""
    
    lines = []
    for line in text.split('\n'):
        line = re_img_md.sub('', line).strip()
        if not line or re_figure.match(line):
            continue
        lines.append(line)
    
    return '\n'.join(lines)

node_content = []
if Path(NODE_FILE).exists():
    with open(NODE_FILE, "r", encoding="utf-8") as f:
        for line in f:
            data = json.loads(line.strip())
            node_content.append(data["text"])

special_path = Path(SPECIAL_FILE)
if special_path.exists():
    special_content = process_special_file(special_path)
    all_content = node_content + special_content

    with open(MERGED_FILE, "w", encoding="utf-8") as fout:
        for content in all_content:
            fout.write(json.dumps({"text": content}, ensure_ascii=False) + "\n")
print(f"output dataset is: {MERGED_FILE}")

output dataset is: node1.jsonl


In [1]:
import json
import re
from pathlib import Path

INPUT_FILE = Path("node1.jsonl")
OUTPUT_FILE = Path("node2.jsonl")

# filter the paragraphs which only have simple title
def filter_single_title_nodes():
    filtered_count = 0
    total_count = 0
    useful_count = 0
    
    with INPUT_FILE.open("r", encoding="utf-8") as fin, \
         OUTPUT_FILE.open("w", encoding="utf-8") as fout:
        
        for line in fin:
            total_count += 1
            data = json.loads(line.strip())
            text = data["text"]

            if has_actual_content(text):
                cleaned_text = remove_little_title_tags(text)
                useful_count += 1
                fout.write(json.dumps({
                    "id": f"{useful_count}",  
                    "text": cleaned_text
                }, ensure_ascii=False) + "\n")
            else:
                filtered_count += 1
    
    print(f"formal nodes: {total_count}")
    print(f"useful nodes: {useful_count}")
    print(f"filtered nodes: {filtered_count}")
    print(f"output file: {OUTPUT_FILE}")

def has_actual_content(text: str) -> bool:
    lines = text.split('\n')
    if len(lines) <= 1:
        return False
    for line in lines[1:]:
        if line.strip() and not line.startswith('<little_title>'):
            return True
    return False

def remove_little_title_tags(text: str) -> str:
    return re.sub(r'<little_title>', '', text)

if __name__ == "__main__":
    filter_single_title_nodes()

formal nodes: 1213
useful nodes: 1135
filtered nodes: 78
output file: node2.jsonl


In [34]:
import json
from transformers import AutoTokenizer
from pathlib import Path
import numpy as np

tokenizer = AutoTokenizer.from_pretrained("Qwen-0.6B-local", trust_remote_code=True)

token_counts = []
with open("node2.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        data = json.loads(line.strip())
        tokens = tokenizer.encode(data["text"], add_special_tokens=False)
        token_counts.append(len(tokens))

print(f"total paragraphs: {len(token_counts)}")
print(f"average tokens in each paragraph: {np.mean(token_counts):.1f}")

total paragraphs: 1135
average tokens in each paragraph: 1972.2


In [1]:
import json
from transformers import AutoTokenizer
from pathlib import Path
import numpy as np

tokenizer = AutoTokenizer.from_pretrained("deepseek-v2", trust_remote_code=True)

token_counts = []
with open("node2.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        data = json.loads(line.strip())
        tokens = tokenizer.encode(data["text"], add_special_tokens=False)
        token_counts.append(len(tokens))

print(f"total paragraphs: {len(token_counts)}")
print(f"average tokens in each paragraph: {np.mean(token_counts):.1f}")

Token indices sequence length is longer than the specified maximum sequence length for this model (22632 > 16384). Running this sequence through the model will result in indexing errors


total paragraphs: 1135
average tokens in each paragraph: 1940.7


In [7]:
import json

def reassign_sequential_ids(input_file, output_file, start_id=1, id_key='id'):
    current_id = start_id
    
    with open(input_file, 'r', encoding='utf-8') as f_in, \
         open(output_file, 'w', encoding='utf-8') as f_out:
        
        for line in f_in:
            if line.strip():  
                data = json.loads(line.strip())
         
                data[id_key] = current_id
                current_id += 1

                f_out.write(json.dumps(data, ensure_ascii=False) + '\n')
    
    print(f"new id: {start_id} - {current_id - 1}")

if __name__ == "__main__":
    input_file = "node3.jsonl"    
    output_file = "node4.jsonl" 
    
    reassign_sequential_ids(input_file, output_file)
    print(f"Saved new file as {output_file}")

new id: 1 - 1156
Saved new file as node4.jsonl


In [2]:
import json

def reassign_sequential_ids(input_file, output_file, start_id=1, id_key='id'):
    current_id = start_id
    
    with open(input_file, 'r', encoding='utf-8') as f_in, \
         open(output_file, 'w', encoding='utf-8') as f_out:
        
        for line in f_in:
            if line.strip():  
                data = json.loads(line.strip())
         
                data[id_key] = str(current_id)
                current_id += 1

                f_out.write(json.dumps(data, ensure_ascii=False) + '\n')
    
    print(f"new id: {start_id} - {current_id - 1}")

if __name__ == "__main__":
    input_file = "node6.jsonl"    
    output_file = "node7.jsonl" 
    
    reassign_sequential_ids(input_file, output_file)
    print(f"Saved new file as {output_file}")

new id: 1 - 1108
Saved new file as node7.jsonl


In [1]:
import json
from transformers import AutoTokenizer
from pathlib import Path
import numpy as np

tokenizer = AutoTokenizer.from_pretrained("deepseek-v2", trust_remote_code=True)

THRESHOLD = 10000  

token_counts = []
long_paragraphs = []  

with open("node2.jsonl", "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        data = json.loads(line.strip())
        tokens = tokenizer.encode(data["text"], add_special_tokens=False)
        token_count = len(tokens)
        token_counts.append(token_count)

        if token_count > THRESHOLD:
            long_paragraphs.append({
                "index": i,
                "token_count": token_count
            })

print(f"Total paragraphs: {len(token_counts)}")
print(f"Average tokens per paragraph: {np.mean(token_counts):.1f}")
print(f"Paragraphs exceeding {THRESHOLD} tokens: {len(long_paragraphs)}")

if long_paragraphs:
    print("\nDetails of long paragraphs:")
    for para in long_paragraphs:
        print(f"Index: {para['index']}, Tokens: {para['token_count']}")


Token indices sequence length is longer than the specified maximum sequence length for this model (22632 > 16384). Running this sequence through the model will result in indexing errors


Total paragraphs: 1135
Average tokens per paragraph: 1940.7
Paragraphs exceeding 10000 tokens: 7

Details of long paragraphs:
Index: 3, Tokens: 10926
Index: 53, Tokens: 10370
Index: 189, Tokens: 10144
Index: 190, Tokens: 10588
Index: 207, Tokens: 10659
Index: 213, Tokens: 22632
Index: 215, Tokens: 13929


In [15]:
import json
from transformers import AutoTokenizer
from pathlib import Path
import numpy as np

tokenizer = AutoTokenizer.from_pretrained("deepseek-v2", trust_remote_code=True)

THRESHOLD = 10000  

token_counts = []
long_paragraphs = []  

with open("node4.jsonl", "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        data = json.loads(line.strip())
        tokens = tokenizer.encode(data["text"], add_special_tokens=False)
        token_count = len(tokens)
        token_counts.append(token_count)

        if token_count > THRESHOLD:
            long_paragraphs.append({
                "index": i,
                "token_count": token_count
            })

print(f"Total paragraphs: {len(token_counts)}")
print(f"Average tokens per paragraph: {np.mean(token_counts):.1f}")
print(f"Paragraphs exceeding {THRESHOLD} tokens: {len(long_paragraphs)}")

if long_paragraphs:
    print("\nDetails of long paragraphs:")
    for para in long_paragraphs:
        print(f"Index: {para['index']}, Tokens: {para['token_count']}")

Total paragraphs: 1156
Average tokens per paragraph: 1900.5
Paragraphs exceeding 10000 tokens: 0


In [10]:
import json
from transformers import AutoTokenizer
from pathlib import Path
import numpy as np

tokenizer = AutoTokenizer.from_pretrained("deepseek-v2", trust_remote_code=True)

THRESHOLD_LONG = 10000  
THRESHOLD_SHORT = 150

token_counts = []
long_paragraphs = []  
short_paragraphs = []

with open("node4.jsonl", "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        data = json.loads(line.strip())
        tokens = tokenizer.encode(data["text"], add_special_tokens=False)
        token_count = len(tokens)
        token_counts.append(token_count)

        if token_count > THRESHOLD_LONG:
            long_paragraphs.append({
                "index": i,
                "token_count": token_count
            })
        
        if token_count < THRESHOLD_SHORT:
            short_paragraphs.append({
                "index": i,
                "token_count": token_count
            })

print(f"Total paragraphs: {len(token_counts)}")
print(f"Average tokens per paragraph: {np.mean(token_counts):.1f}")
print(f"Paragraphs exceeding {THRESHOLD_LONG} tokens: {len(long_paragraphs)}")
print(f"Paragraphs below {THRESHOLD_SHORT} tokens: {len(short_paragraphs)}")

if long_paragraphs:
    print("\nDetails of long paragraphs:")
    for para in long_paragraphs:
        print(f"Index: {para['index']}, Tokens: {para['token_count']}")

if short_paragraphs:
    print("\nDetails of short paragraphs:")
    for para in short_paragraphs:
        print(f"Index: {para['index']}, Tokens: {para['token_count']}")

Total paragraphs: 1156
Average tokens per paragraph: 1900.5
Paragraphs exceeding 10000 tokens: 0
Paragraphs below 150 tokens: 48

Details of short paragraphs:
Index: 12, Tokens: 71
Index: 16, Tokens: 71
Index: 26, Tokens: 126
Index: 38, Tokens: 98
Index: 48, Tokens: 98
Index: 53, Tokens: 56
Index: 73, Tokens: 99
Index: 77, Tokens: 45
Index: 88, Tokens: 55
Index: 99, Tokens: 99
Index: 104, Tokens: 51
Index: 120, Tokens: 70
Index: 127, Tokens: 79
Index: 137, Tokens: 62
Index: 149, Tokens: 105
Index: 324, Tokens: 87
Index: 361, Tokens: 79
Index: 390, Tokens: 146
Index: 409, Tokens: 60
Index: 434, Tokens: 86
Index: 451, Tokens: 99
Index: 453, Tokens: 113
Index: 455, Tokens: 143
Index: 479, Tokens: 105
Index: 487, Tokens: 139
Index: 509, Tokens: 108
Index: 521, Tokens: 114
Index: 534, Tokens: 136
Index: 538, Tokens: 101
Index: 629, Tokens: 59
Index: 656, Tokens: 109
Index: 671, Tokens: 107
Index: 697, Tokens: 79
Index: 703, Tokens: 48
Index: 706, Tokens: 102
Index: 712, Tokens: 127
Index: 7

In [3]:
import json
from transformers import AutoTokenizer
from pathlib import Path
import numpy as np

tokenizer = AutoTokenizer.from_pretrained("deepseek-v2", trust_remote_code=True)

THRESHOLD_LONG = 10000  
THRESHOLD_SHORT = 150

token_counts = []
long_paragraphs = []  
short_paragraphs = []

with open("node6.jsonl", "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        data = json.loads(line.strip())
        tokens = tokenizer.encode(data["text"], add_special_tokens=False)
        token_count = len(tokens)
        token_counts.append(token_count)

        if token_count > THRESHOLD_LONG:
            long_paragraphs.append({
                "index": i,
                "token_count": token_count
            })
        
        if token_count < THRESHOLD_SHORT:
            short_paragraphs.append({
                "index": i,
                "token_count": token_count
            })

print(f"Total paragraphs: {len(token_counts)}")
print(f"Average tokens per paragraph: {np.mean(token_counts):.1f}")
print(f"Paragraphs exceeding {THRESHOLD_LONG} tokens: {len(long_paragraphs)}")
print(f"Paragraphs below {THRESHOLD_SHORT} tokens: {len(short_paragraphs)}")

if long_paragraphs:
    print("\nDetails of long paragraphs:")
    for para in long_paragraphs:
        print(f"Index: {para['index']}, Tokens: {para['token_count']}")

if short_paragraphs:
    print("\nDetails of short paragraphs:")
    for para in short_paragraphs:
        print(f"Index: {para['index']}, Tokens: {para['token_count']}")

Total paragraphs: 1108
Average tokens per paragraph: 1977.5
Paragraphs exceeding 10000 tokens: 0
Paragraphs below 150 tokens: 0
